# Inventaire du corpus — T0
Comptages en lecture seule via les APIs officielles.

In [1]:
import json, time, urllib.request, urllib.parse

UA = {"User-Agent": "P13-POC-inventaire/0.1 (richard.hugou@gmail.com)"}

def api(base, params, retries=4):
    url = base + "?" + urllib.parse.urlencode(params)
    for i in range(retries):
        try:
            req = urllib.request.Request(url, headers=UA)
            out = json.load(urllib.request.urlopen(req, timeout=30))
            time.sleep(0.6)
            return out
        except urllib.error.HTTPError as e:
            if e.code == 429 and i < retries - 1:
                time.sleep(20)
                continue
            raise

## Référentiel d'ouvertures — `lichess-org/chess-openings`

In [2]:
counts = {}
for f in "abcde":
    url = f"https://raw.githubusercontent.com/lichess-org/chess-openings/master/{f}.tsv"
    raw = urllib.request.urlopen(urllib.request.Request(url, headers=UA)).read().decode()
    counts[f"{f}.tsv"] = len(raw.strip().split("\n")) - 1  # moins l'en-tête
total_referentiel = sum(counts.values())
print(counts)
print("Total :", total_referentiel, "ouvertures nommées")

{'a.tsv': 817, 'b.tsv': 772, 'c.tsv': 1250, 'd.tsv': 614, 'e.tsv': 357}
Total : 3810 ouvertures nommées


## Wikipédia FR — arbre `Catégorie:Ouverture d'échecs`

In [3]:
FR = "https://fr.wikipedia.org/w/api.php"

def collect_articles(cmtitle, depth=0, seen_cats=None, articles=None):
    if seen_cats is None: seen_cats, articles = set(), set()
    if cmtitle in seen_cats or depth > 3: return articles
    seen_cats.add(cmtitle)
    cont = {}
    while True:
        r = api(FR, {"action": "query", "list": "categorymembers", "cmtitle": cmtitle,
                     "cmlimit": "500", "cmtype": "page|subcat", "format": "json", **cont})
        for m in r["query"]["categorymembers"]:
            if m["ns"] == 14 and "Projet:" not in m["title"] and "Portail:" not in m["title"]:
                collect_articles(m["title"], depth + 1, seen_cats, articles)
            elif m["ns"] == 0:
                articles.add(m["title"])
        if "continue" in r: cont = r["continue"]
        else: break
    return articles

articles_fr = collect_articles("Catégorie:Ouverture d'échecs")
total_wikipedia_fr = len(articles_fr)
print(total_wikipedia_fr, "articles uniques (dédupliqués par titre)")

225 articles uniques (dédupliqués par titre)


## Wikibooks EN — pages `Chess Opening Theory*`

In [4]:
EN = "https://en.wikibooks.org/w/api.php"
total_wikibooks_en, cont = 0, {}
while True:
    r = api(EN, {"action": "query", "list": "allpages", "apprefix": "Chess Opening Theory",
                 "aplimit": "500", "format": "json", **cont})
    total_wikibooks_en += len(r["query"]["allpages"])
    if "continue" in r: cont = r["continue"]
    else: break
print(total_wikibooks_en, "pages")

3026 pages


## Synthèse

In [5]:
import datetime
print(f"Inventaire du {datetime.date.today()}")
print(f"Référentiel chess-openings : {total_referentiel} ouvertures nommées")
print(f"Wikipédia FR (ouvertures)  : {total_wikipedia_fr} articles uniques")
print(f"Wikibooks EN (théorie)     : {total_wikibooks_en} pages")

Inventaire du 2026-08-22
Référentiel chess-openings : 3810 ouvertures nommées
Wikipédia FR (ouvertures)  : 225 articles uniques
Wikibooks EN (théorie)     : 3026 pages
